<a href="https://colab.research.google.com/github/sibot89/RAG-systems/blob/main/RAG_system_HuggingFace%26FAISS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
!pip install --upgrade langchain
!pip install transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.9/136.9 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 15.3 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.8
    Uninstalling langchain-core-1.4.8:
      Successfully uninstalled langchain-core-1.4.8
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.11
    Uninstalling langchain-1.3.11:
      Successfully uninstalled langchain-1.3.11


In [5]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 63.5 MB/s eta 0:00:00


In [7]:
pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 66.2 MB/s eta 0:00:00


In [8]:
from transformers import pipeline
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import fitz

In [19]:
#pdf loading and chunking
def load_pdf(file_path):
  doc = fitz.open(file_path)
  text = ""
  for page_num in range(doc.page_count):
    page = doc.load_page(page_num)
    text += page.get_text()
  return text

import kagglehub

# Download latest version
#path = kagglehub.dataset_download("azrilbagas/pdf-file-rag")

path = "/content/RAG_sample-pdf/Deep Learning with Python by François - datayad.com.pdf"
pdf_text = load_pdf(path)

def recursive_chunk(text, max_chunk_size=500):
  chunks = []
  while len(text) > max_chunk_size:
    chunk = text[:max_chunk_size]
    chunks.append(chunk)
    text = text[max_chunk_size:]
  if text:
    chunks.append(text)
  return chunks

document_chunks = recursive_chunk(pdf_text)

#print(document_chunks)

In [20]:
#Embedding Generation and Indexing
model = SentenceTransformer('all-miniLM-L6-v2')

chunk_embeddings = model.encode(document_chunks)

chunk_embeddings_np = np.array(chunk_embeddings).astype('float32')

index = faiss.IndexFlatL2(chunk_embeddings_np.shape[1])
index.add(chunk_embeddings_np)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [23]:
#Query Processing and Retrival
generator = pipeline("text-generation", model='gpt2')

def retrive_and_generate(query):

  query_embedding = model.encode([query])[0].astype('float32')

  _, indices = index.search(np.array([query_embedding]), k=3)

  relevant_chunks = [document_chunks[i] for i in indices[0]]

  context = "".join(relevant_chunks)

  responce = generator(f"{query} {context}", max_length=100, num_return_sequences=1)
  return responce[0]["generated_text"]

query = "What is deep learning?"
responce = retrive_and_generate(query)
print(responce)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it

What is deep learning? related model is logistic regression (logreg for short), which is sometimes
considered to be the “Hello World” of modern machine learning. Don’t be misled by
its name—logreg is a classification algorithm rather than a regression algorithm. Much
14
CHAPTER 1
What is deep learning?
like Naive Bayes, logreg predates computing by a long time, yet it’s still useful to this
day, thanks to its simple and versatile nature. It’s often the first thing a data scientist
will try on a dataset to get a feel flusions
431
ix
contents
 preface
xvii
acknowledgments
xix
about this book
xx
about the author
xxiii
about the cover illustration
xxiv
1 
What is deep learning?
1
1.1
Artificial intelligence, machine learning, and deep 
learning
2
Artificial intelligence
2
■Machine learning
3
■Learning 
rules and representations from data
4
■The “deep” in “deep 
learning”
7
■Understanding how deep learning works, in 
three figures
8
■What deep learning has achieved so far
10
Don’t believe t